# Visual Caption Generation (Qwen2.5-VL)

Generates a detailed caption for every figure MinerU2.5 extracted from a
PDF — photos, labeled diagrams, charts, and tables all go through
**Qwen2.5-VL**, since the goal here is a detailed, well-grounded
description rather than a clinical/diagnostic reading.

For diagrams MinerU2.5 tags as `sub_type == "text_image"`, it also OCRs
the leader-line labels into the `content` field. Those labels are passed
into the prompt as grounding context, so the model transcribes and places
the existing labels instead of re-reading the image from scratch (this is
what previously tripped up a photo-specialized model into hallucinating a
fake CT scan reading for a labeled mouth-anatomy diagram).

**Hardware note:** runs on Apple Silicon (MPS, no CUDA). Uses bfloat16 and
loads to CPU before `.to("mps")` — torch's MPS backend has unresolved
SIGSEGVs in its fp16 cast kernel and in device_map-based loading (see
`src/captioning/qwen_vl.py` docstring for issue links).

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from captioning.qwen_vl import QwenVLCaptioner

PROJECT_ROOT

/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PosixPath('/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student')

## Load MinerU2.5 output for one PDF

In [2]:
from ingestion.clean_content_list import build_clean_content_list

PDF_STEM = "Anatomy of Face and Oral Cavity - Basic of DEMN.pdf"
OUTPUT_DIR = PROJECT_ROOT / "output" / PDF_STEM / "hybrid_auto"
CONTENT_LIST_V2_PATH = OUTPUT_DIR / f"{PDF_STEM}_content_list_v2.json"
CONTENT_LIST_PATH = build_clean_content_list(CONTENT_LIST_V2_PATH)

with open(CONTENT_LIST_PATH) as f:
    content_list = json.load(f)

visual_items = [
    item for item in content_list
    if item.get("type") in {"image", "chart", "table", "diagram"} and item.get("img_path")
]
len(visual_items), visual_items[0]

(102,
 {'type': 'image',
  'img_path': 'images/b6b3baa9cb90ab1ea5bf18406845ade4c753c010a64d101e570d1f0920cc3689.jpg',
  'image_caption': [],
  'image_footnote': [],
  'content': 'Anatomical illustration of human head and neck muscles, showing detailed anatomical structures without any text or labels.',
  'sub_type': 'natural_image',
  'bbox': [0, 0, 500, 998],
  'page_idx': 0})

## Smoke test on one labeled diagram

In [3]:
captioner = QwenVLCaptioner()

# Prefer a text_image (labeled diagram) sample if this PDF has one.
sample = next((i for i in visual_items if i.get("sub_type") == "text_image"), visual_items[0])

image_path = OUTPUT_DIR / sample["img_path"]
caption = captioner.caption(str(image_path), labels=sample.get("content") or None)

print(json.dumps({
    "image_path": str(image_path),
    "content_type": sample["type"],
    "sub_type": sample.get("sub_type"),
    "caption": caption,
}, indent=2))

Loading weights: 100%|██████████| 729/729 [00:00<00:00, 10003.56it/s]


{
  "image_path": "/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf/hybrid_auto/images/cefe146f8b2e0535e34708f615e93974134dfe48bb0fe426ccea0ad5d9ed9a26.jpg",
  "content_type": "image",
  "sub_type": "text_image",
  "caption": "The image is a detailed anatomical illustration of the human digestive system. It shows a side view of a person with internal organs labeled. Here's a breakdown of the visual elements:\n\n1. **Top-Left Region:**\n   - The text \"Stive System\" is written at the top-left corner.\n   - Below that, there are labels for the mouth and tongue, pointing to the oral cavity and tongue respectively.\n\n2. **Top-Right Region:**\n   - Labels for salivary glands: Parotid gland, Sublingual gland, and Submandibular gland.\n   - These glands are shown near the mouth area.\n\n3. **Middle Region:**\n   - Esophagus is labeled and points to the tube connecting the pharynx to th

## Batch run over all visuals in the PDF

In [ ]:
from tqdm.auto import tqdm

results = []

for item in tqdm(visual_items, desc="Captioning visuals"):
    image_path = OUTPUT_DIR / item["img_path"]
    caption = captioner.caption(str(image_path), labels=item.get("content") or None)
    results.append({
        "image_path": str(image_path),
        "content_type": item["type"],
        "sub_type": item.get("sub_type"),
        "caption": caption,
    })

captioner.unload()
len(results)

Captioning visuals:   1%|          | 1/102 [00:32<55:18, 32.86s/it]

In [ ]:
results_path = OUTPUT_DIR / f"{PDF_STEM}_stage1_captions.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

results_path

PosixPath('/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf/hybrid_auto/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_stage1_captions.json')

## Unify captions with document text

Merges Stage 1 captions back into the document, replacing each figure
with a `[FIGURE:item_id]` marker + caption, inlined in original reading
order. `item_id` (`{pdf_stem}#p{page_idx}#{index}`) is what ties an image
to its page and its caption together — see `src/ingestion/unify.py`.

In [ ]:
from ingestion.unify import build_unified_items, render_unified_text, render_unified_markdown, captions_by_item_id

captions = captions_by_item_id(content_list, PDF_STEM, results)
unified_items = build_unified_items(content_list, PDF_STEM, captions)
unified_text = render_unified_text(unified_items)

print(f"{len(unified_items)} unified items, {len(unified_text)} chars")
print(unified_text[:1500])

348 unified items, 38174 chars
THE DIGESTIVE SYSTEM:FACE AND ORALCAVITY

Basic of DEMN System

THE DIGESTIVE SYSTEM

<sup>▪</sup> Digestive System: The system whose function it is to break down foods into molecules small enough to enter body cells.

<sup>▪</sup> Allows the body to ingest & digest proteins, fats & carbohydrates & absorb them into the bloodstream & lymph to be taken to body cells for metabolism & conversion to ATP.

<sup>▪</sup> Gastrointestinal (GI) Tract aka Alimentary Canal: A continuous tube that extends from the mouth to the anus.

<sup>▪</sup> Tonus: The sustained muscular contraction of the GI tract walls that helps to move food along.

O

0

FACE AREA

THE FACE

Boundaries

- Extends superiorly to the hair line, inferiorly to the chin and base of mandible, and on each side to auricle
- <sup>▪</sup> Forehead is common to both scalp and face.
- <sup>▪</sup> Very vascular <sup>→</sup> Face blush and blanch.
- <sup>▪</sup> Wounds of face bleed profusely but heal rapi

In [ ]:
unified_text_path = OUTPUT_DIR / f"{PDF_STEM}_unified_text.txt"
with open(unified_text_path, "w") as f:
    f.write(unified_text)

unified_text_path

PosixPath('/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf/hybrid_auto/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_unified_text.txt')

In [ ]:
unified_markdown = render_unified_markdown(unified_items)

unified_md_path = OUTPUT_DIR / f"{PDF_STEM}_unified_text.md"
with open(unified_md_path, "w") as f:
    f.write(unified_markdown)

unified_md_path

PosixPath('/Users/michaeleko/Documents/Projects/challenge-2/intelligent-tutoring-system-for-medical-student/output/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf/hybrid_auto/Anatomy of Face and Oral Cavity - Basic of DEMN.pdf_unified_text.md')